In [0]:
spark.conf.set("fs.azure.account.auth.type.proddatalakefordatabrick.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.proddatalakefordatabrick.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.proddatalakefordatabrick.dfs.core.windows.net", "8b03b30e-c838-4568-a41e-a801634f0e96")
spark.conf.set("fs.azure.account.oauth2.client.secret.proddatalakefordatabrick.dfs.core.windows.net", dbutils.secrets.get(scope="darshanscope", key="app-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.proddatalakefordatabrick.dfs.core.windows.net", "https://login.microsoftonline.com/5ff33f54-37cd-4d6d-9156-f6c00f164999/oauth2/token")

In [0]:
df = spark.read.table("proddatabricksdemo.silver.order")

In [0]:
df_dimcus = spark.sql("select DimCustomerKey, customer_id as Dim_customer_id from proddatabricksdemo.gold.dimcustomer")

df_dimpro = spark.sql("select DimProductsKey,product_id as DimProducts_id from proddatabricksdemo.gold.dimproducts")


In [0]:
df_fact = (
    df.join(
        df_dimcus,
        df["customer_id"] == df_dimcus["Dim_customer_id"],
        "left"
    )
    .join(
        df_dimpro,
        df["product_id"] == df_dimpro["DimProducts_id"],
        "left"
    )
)

In [0]:
df_fact_new = df_fact.drop(
    "DimProducts_id",
    "Dim_customer_id",
    "customer_id",
    "product_id"
)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("proddatabricksdemo.gold.FactOrders"):
    
    dlt_obj = DeltaTable.forName(spark, "proddatabricksdemo.gold.FactOrders")

    dlt_obj.alias("trg").merge(df_fact_new.alias("src"), "trg.order_id = src.order_id AND trg.DimCustomerKey = src.DimCustomerKey AND trg.DimProductsKey = src.DimProductsKey")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    df_fact_new.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable("proddatabricksdemo.gold.orders")

In [0]:
df_fact_new.write.format("delta").mode("overwrite").save("abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/gold/order")